# Vision QA — clasificator de familie și verdict

**Scop.** Lansează antrenarea pe un singur GPU (`ml.g5.xlarge` spot), citește metricele și
verifică modelul pe câteva imagini de test. Modelul are două capete peste același ResNet18:
familia piesei (șase clase) și verdictul PASS/REJECT.

**Intrare.** Datasetul de piese din S3 (8.000 de imagini, 2.000 de piese).
**Ieșire.** `s3://cncdt-dev-datalake-864981730944/models/vision-qa/vision-qa-v2/<job>/output/model.tar.gz`.

Antrenarea durează ~15–20 de minute, cu limita de o oră în job. Nu rămâne nimic pornit.


## 1. Mediu, identitate și imaginea de container


In [ ]:
from pathlib import Path
import io, json, tarfile, time

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

REGION = "eu-central-1"
DL = "cncdt-dev-datalake-864981730944"
CORPUS = "cncdt-dev-training-data-864981730944"
PARTS = "training-data/parts/dt=2026-10-18/run-20261018T201942/"
VERSION = "vision-qa-v2"
IMAGE = (f"763104351884.dkr.ecr.{REGION}.amazonaws.com/pytorch-training:"
         "2.4.0-gpu-py311-cu124-ubuntu22.04-sagemaker")
INFERENCE_IMAGE = (f"763104351884.dkr.ecr.{REGION}.amazonaws.com/pytorch-inference:"
                   "2.4.0-cpu-py311-ubuntu22.04-sagemaker-v1.34")

s3 = boto3.client("s3", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)
arn = boto3.client("sts", region_name=REGION).get_caller_identity()["Arn"]
ROLE = f"arn:aws:iam::{arn.split(':')[4]}:role/{arn.split('/')[1]}"
CODE = Path.home() / "cnc-code"
bundle = Path("/tmp/cncdt-code.tgz")
s3.download_file(DL, "notebooks/studio/code/cncdt-code.tgz", str(bundle))
CODE.mkdir(exist_ok=True)
tarfile.open(bundle).extractall(CODE)
print(ROLE, "| cod:", CODE)


## 2. Împachetează scriptul de antrenare

SageMaker script-mode primește un tarball cu `train.py` la rădăcină. Fișierele vin din pachetul
de cod al proiectului, deci aceeași versiune ajunge în job și în repo. `vision_model.py` este
arhitectura comună antrenării și inferenței.


In [ ]:
files = [
    ("ml/training/vision_qa/train.py", "train.py"),
    ("ml/inference/vision_qa/inference.py", "inference.py"),
    ("ml/inference/vision_qa/vision_inference_core.py", "vision_inference_core.py"),
    ("ml/inference/vision_qa/vision_model.py", "vision_model.py"),
]
missing = [relative for relative, _ in files if not (CODE / relative).is_file()]
if missing:
    raise SystemExit("lipsește din pachetul de cod: " + ", ".join(missing))

tarball = Path("/tmp/vision-sourcedir.tar.gz")
with tarfile.open(tarball, "w:gz") as tar:
    for relative, arcname in files:
        tar.add(CODE / relative, arcname=arcname)
s3.upload_file(str(tarball), DL, "code/vision_qa/sourcedir.tar.gz")
print("urcat", tarball.stat().st_size, "bytes")


## 3. Lansează antrenarea


In [ ]:
JOB = "cncdt-vision-qa-" + time.strftime("%Y%m%d-%H%M%S")
sm.create_training_job(
    TrainingJobName=JOB,
    AlgorithmSpecification={"TrainingImage": IMAGE, "TrainingInputMode": "File"},
    RoleArn=ROLE,
    InputDataConfig=[{
        "ChannelName": "train",
        "DataSource": {"S3DataSource": {
            "S3DataType": "S3Prefix",
            "S3Uri": f"s3://{CORPUS}/{PARTS}",
            "S3DataDistributionType": "FullyReplicated"}},
        "ContentType": "application/x-image",
        "InputMode": "File"}],
    OutputDataConfig={"S3OutputPath": f"s3://{DL}/models/vision-qa/{VERSION}/"},
    ResourceConfig={"InstanceType": "ml.g5.xlarge", "InstanceCount": 1,
                    "VolumeSizeInGB": 30},
    StoppingCondition={"MaxRuntimeInSeconds": 3600, "MaxWaitTimeInSeconds": 5400},
    HyperParameters={
        "sagemaker_program": "train.py",
        "sagemaker_submit_directory": f"s3://{DL}/code/vision_qa/sourcedir.tar.gz",
        "epochs": "10", "batch_size": "64", "lr": "0.0003",
        "image_size": "224", "workers": "4", "family_loss_weight": "0.5"},
    EnableManagedSpotTraining=True,
)
print("job:", JOB)


## 4. Așteaptă și citește metricele


In [ ]:
sm.get_waiter("training_job_completed_or_stopped").wait(
    TrainingJobName=JOB, WaiterConfig={"Delay": 30, "MaxAttempts": 120})
desc = sm.describe_training_job(TrainingJobName=JOB)
print(desc["TrainingJobStatus"], desc.get("SecondaryStatus"))
assert desc["TrainingJobStatus"] == "Completed", desc.get("FailureReason")

model_key = f"models/vision-qa/{VERSION}/{JOB}/output/model.tar.gz"
s3.download_file(DL, model_key, "/tmp/vision-model.tar.gz")
with tarfile.open("/tmp/vision-model.tar.gz") as tar:
    tar.extractall("/tmp/vision-model")
metrics = json.loads(Path("/tmp/vision-model/metrics.json").read_text(encoding="utf-8"))
DEPLOY_KEY = f"models/vision-qa/{VERSION}/model.tar.gz"
s3.upload_file("/tmp/vision-model.tar.gz", DL, DEPLOY_KEY)
deployment = {"version": VERSION, "model_data_url": f"s3://{DL}/{DEPLOY_KEY}",
              "inference_image": INFERENCE_IMAGE, "contract": "cnc.qa.snapshot.v1",
              "heads": ["family", "verdict"], "families": metrics["families"],
              "threshold": metrics["threshold"]}
s3.put_object(Bucket=DL, Key=f"models/vision-qa/{VERSION}/deployment.json",
              Body=json.dumps(deployment, indent=2, ensure_ascii=False).encode())
print(json.dumps(deployment, indent=1, ensure_ascii=False))
print(json.dumps({k: metrics[k] for k in ("best_epoch", "threshold", "test", "test_family")},
                 indent=1, ensure_ascii=False))


## 5. Matricea de confuzie pentru verdict și pentru familie


In [ ]:
def plot_confusion(cm, labels, title):
    cm = np.asarray(cm)
    size = len(labels)
    fig, ax = plt.subplots(figsize=(max(4.2, size * 0.85), max(3.6, size * 0.75)))
    ax.imshow(cm, cmap="Blues")
    for i in range(size):
        for j in range(size):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                    fontsize=9 if size > 2 else 11)
    ax.set_xticks(range(size), labels, rotation=35 if size > 2 else 0,
                  ha="right" if size > 2 else "center")
    ax.set_yticks(range(size), labels)
    ax.set_xlabel("prezis"); ax.set_ylabel("real"); ax.set_title(title)
    fig.tight_layout()
    return fig

verdict_labels = ["PASS", "REJECT"]
fig = plot_confusion(metrics["val"]["confusion"], verdict_labels,
                     f"verdict val · prag {metrics['threshold']}")
fig.savefig("/tmp/vision-val-confusion.png", bbox_inches="tight", dpi=150)
fig = plot_confusion(metrics["test"]["confusion"], verdict_labels, "verdict test")
fig.savefig("/tmp/vision-test-confusion.png", bbox_inches="tight", dpi=150)
fig = plot_confusion(metrics["test_family"]["confusion"], metrics["families"],
                     f"familie test · acuratețe {metrics['test_family']['accuracy']:.3f}")
fig.savefig("/tmp/vision-family-confusion.png", bbox_inches="tight", dpi=150)

defect = pd.DataFrame(metrics["test_by_defect"]).T
defect = defect[defect["recall_reject"].notna()].copy()
defect["recall_reject"] = defect["recall_reject"].astype(float)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
defect["recall_reject"].sort_values().plot.barh(ax=ax, color="#d62728")
ax.set_xlim(0, 1.02); ax.set_xlabel("recall pe REJECT"); ax.set_title("Test, pe tip de defect")
fig.tight_layout()
fig.savefig("/tmp/vision-defect-recall.png", bbox_inches="tight", dpi=150)

for name in ("vision-val-confusion", "vision-test-confusion",
             "vision-family-confusion", "vision-defect-recall"):
    s3.upload_file(f"/tmp/{name}.png", DL, f"models/vision-qa/{VERSION}/plots/{name}.png")

display(pd.DataFrame(metrics["test_by_family"]).T)
display(pd.DataFrame(metrics["test_family"]["recall_by_family"], index=["recall_familie"]).T)
print("acuratețe familie (test):", metrics["test_family"]["accuracy"],
      "| macro recall:", metrics["test_family"]["macro_recall"])


## 6. Verificare pe câteva imagini de test


In [ ]:
try:
    import sys
    import torch
    from torchvision import transforms
except Exception as error:
    print("torch indisponibil în kernel; sari peste verificare:", error)
else:
    sys.path.insert(0, str(CODE / "ml/inference/vision_qa"))
    from vision_model import MultiHeadNet

    bundle = torch.load("/tmp/vision-model/model.pth", map_location="cpu", weights_only=False)
    model = MultiHeadNet(len(bundle["families"]), len(bundle["classes"]))
    model.load_state_dict(bundle["state_dict"])
    model.eval()

    prep = transforms.Compose([
        transforms.Resize((bundle["image_size"], bundle["image_size"])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    body = s3.get_object(Bucket=CORPUS, Key=PARTS + "images-test.jsonl")["Body"].read().decode()
    rows = [json.loads(line) for line in body.splitlines() if line]
    picks = ([r for r in rows if r["verdict"] == "PASS"][:3] +
             [r for r in rows if r["verdict"] == "REJECT"][:3])

    fig, axes = plt.subplots(2, 3, figsize=(12, 7.5))
    for ax, row in zip(axes.ravel(), picks):
        raw = s3.get_object(Bucket=CORPUS, Key=PARTS + row["path"])["Body"].read()
        image = Image.open(io.BytesIO(raw)).convert("RGB")
        with torch.no_grad():
            family_logits, verdict_logits = model(prep(image).unsqueeze(0))
            prob = float(torch.softmax(verdict_logits, dim=1)[0, 1])
            family_prob, family_index = torch.softmax(family_logits, dim=1)[0].max(dim=0)
        verdict = "REJECT" if prob >= bundle["threshold"] else "PASS"
        predicted_family = bundle["families"][int(family_index)]
        ax.imshow(image); ax.axis("off")
        ax.set_title(f"familie {row['family']} → {predicted_family} "
                     f"({float(family_prob):.2f})\n"
                     f"{row['defect']} · real {row['verdict']} → {verdict} "
                     f"(P={prob:.2f})", fontsize=8)
    fig.tight_layout()


## 7. Pasul următor

1. Scrie cardul în `ml/models/vision-qa.md`: date, split, prag, recall pe familie și pe defect,
   plus acuratețea capului de familie.
2. Rulează din nou cu pragul ales dacă recall-ul pe PASS scade sub limita operațională.
3. Artefactul conține `code/inference.py`, `code/vision_inference_core.py` și
   `code/vision_model.py`; pentru simulare completează `vision_qa_model_data_url` cu
   `deployment.json` și pornește endpoint-ul serverless din `stacks/platform`.
4. Ruta de sortare: verdict REJECT → rebut; verdict PASS → compartimentul dat de familia
   prezisă. Sub o încredere de familie stabilită, piesa urmează aceeași cale ca un cadru
   nejugat, adică ruta de rezervă.
